In [1]:
library(ggplot2)
library(dplyr)
# Load the required libraries
library(ggpubr)
# load dplyr and tidyr library

library(tidyr)
library(parallel)
library(future)
library(furrr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
setwd("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/analysis/sRNA_deseq/sRNA_STAR_testing")

In [3]:
data <- read.csv("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/TEtranscriptCount_testing/all_featureCounts_TE_testing.tab", header=FALSE,sep='\t')
data

V1,V2,V3,V4,V5,V6,V7,V8
<chr>,<chr>,<int>,<int>,<chr>,<int>,<dbl>,<chr>
pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts,,NA,NA,,NA,NA,
L1MdTf_II LINE/L1 C57BL_6NJ#1#100 1-2759,C57BL_6NJ#1#100,1,2759,-,2759,0,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts
(A)n Simple_repeat C57BL_6NJ#1#100 2952-2989,C57BL_6NJ#1#100,2952,2989,+,38,0,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts
B1_Mm SINE/Alu C57BL_6NJ#1#100 2995-3123,C57BL_6NJ#1#100,2995,3123,+,129,0,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts
(AACA)n Simple_repeat C57BL_6NJ#1#100 3125-3152,C57BL_6NJ#1#100,3125,3152,+,28,0,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts
(TTTGT)n Simple_repeat C57BL_6NJ#1#100 3445-3466,C57BL_6NJ#1#100,3445,3466,+,22,0,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts
B1_Mm SINE/Alu C57BL_6NJ#1#100 3467-3609,C57BL_6NJ#1#100,3467,3609,-,143,0,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts
(TTTGT)n Simple_repeat C57BL_6NJ#1#100 3610-3623,C57BL_6NJ#1#100,3610,3623,+,14,0,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts
(T)n Simple_repeat C57BL_6NJ#1#100 3624-3663,C57BL_6NJ#1#100,3624,3663,+,40,0,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts


In [4]:
 # Split name column into 
#data <- data %>% separate(V1, c('V9', 'V10','V11','V12'), sep =" ")
#data

split_name <- function(row) {
    separate(row, V1, c('V9', 'V10', 'V11', 'V12'), sep = " ")
}

# Determine the number of cores to use
num_cores <- 10

# Split the data into chunks for each core
chunks <- split(data, cut(1:nrow(data), num_cores, labels = FALSE))

# Process in parallel
results <- mclapply(chunks, function(chunk) split_name(chunk), mc.cores = num_cores)

# Combine results back together
data <- bind_rows(results)

In [5]:
# Group by V8, V9, V10 and sum the V7 values
#new_data <- data %>%
#  group_by(V8, V9, V10) %>%
#  summarise(summed_V7 = sum(V7, na.rm = TRUE)) %>%
#  ungroup()
#new_data
# Use multicore processing if on Unix/Linux, or multiprocess (which is sequential) if on Windows
plan(multicore)
# Split the data into chunks (for this example, let's say 4 chunks)
chunks <- split(data, cut(1:nrow(data), 150, labels = FALSE))

# Process each chunk in parallel
results <- future_map_dfr(chunks, ~ .x %>%
    group_by(V8, V9, V10) %>%
    summarise(summed_V7 = sum(V7, na.rm = TRUE)) %>%
    ungroup())
new_data <- results %>%
  group_by(V8, V9, V10) %>%
  summarise(summed_V7 = sum(summed_V7, na.rm = TRUE)) %>%
  ungroup()
new_data
write.csv(new_data, "TE_testing_repeat.csv", row.names=FALSE)

`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'V8', 'V9'. Yo

V8,V9,V10,summed_V7
<chr>,<chr>,<chr>,<dbl>
,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts,NA,0.00
,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_1500_3000.featureCounts,NA,0.00
,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_200_400.featureCounts,NA,0.00
,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_400_800.featureCounts,NA,0.00
,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_50_100.featureCounts,NA,0.00
,pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_800_1600.featureCounts,NA,0.00
pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts,(A)n,Simple_repeat,7705.18
pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts,(AA)n,Simple_repeat,0.00
pacBio/C57BL_6NJ/C57BL_6NJ-16.5dpc.1_0_100_200.featureCounts,(AAA)n,Simple_repeat,4.06
